# nb07 — почему исследование не сошлось с ботом, и что от edge остаётся при честном сайзинге

**Повод.** Paper-бот за 25 дней (2026-07-08…08-01, 459 сделок) сделал ≈0 при обещанных Sharpe ~3.
`FINDINGS_2026-08-02.md` закрыл прошлую сессию с неразобранным противоречием в портфельном слое
и с четырьмя снятыми выводами. Здесь всё пересчитано с нуля.

**Правила этой тетради.**
1. Окна объявлены до того, как что-либо считалось, и не двигались:
   TRAIN `2024-01-02..2025-06-30`, VALID `2025-07-01..2026-01-31`, TEST `2026-02-01..2026-07-31`.
   Выбор параметров — только по TRAIN. VALID — проверка. TEST открывается один раз в §6.
2. Основная метрика — **деньги на единицу целевого размера**, `mtu = mean(frac·pnl)`.
   Она инвариантна к весу ноги, поэтому расписание не может выиграть просто тем, что
   разворачивает меньше капитала (в эту ловушку упёрлось FINDINGS §5.4).
3. Портфель считается при **фиксированном капитале** (без компаундинга и без DCA): тогда
   ряд доходностей меряет edge, а не рост счёта, и Sharpe/DD сравнимы между конфигурациями.

**Инструменты, на которых всё стоит, проверены отдельно** (nb07 §0): `run_dca` сверен с
замкнутыми формулами, `_portfolio.run_book` сверен с `run_dca` до последнего знака,
v4-якорь `et` совпадает с v3 бит-в-бит.

In [1]:
import sys, re, warnings; sys.path.insert(0, ".")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from _portfolio import (load_v4, window, run_book, stats, build_book_v4, ffrac,
                        check_parity, TRAIN, VALID, TEST)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 50)

PUMP, DUMP = load_v4("pump", "_out"), load_v4("dump", "_out")
BOTCFG = open(r"C:\projects\pump_fade_bot\config.py", encoding="utf-8").read()
BOT120 = sorted(set(re.findall(r'"([A-Z0-9]+USDT)"',
                    BOTCFG.split('"symbols": [', 1)[1].split("],", 1)[0])))
print(f"pump {len(PUMP):6d} events  {PUMP.sym.nunique()} symbols  {PUMP.entry.min().date()}..{PUMP.entry.max().date()}")
print(f"dump {len(DUMP):6d} events  {DUMP.sym.nunique()} symbols  {DUMP.entry.min().date()}..{DUMP.entry.max().date()}")
print(f"bot universe {len(BOT120)} symbols")
for n, w in [("TRAIN", TRAIN), ("VALID", VALID), ("TEST", TEST)]:
    print(f"  {n:5s} {w[0]}..{w[1]}   pump {len(window(PUMP,w)):6d}   dump {len(window(DUMP,w)):6d}")

pump  49833 events  573 symbols  2024-01-02..2026-07-31
dump  20706 events  549 symbols  2024-01-02..2026-07-31
bot universe 120 symbols
  TRAIN 2024-01-02..2025-06-30   pump  18933   dump   6356
  VALID 2025-07-01..2026-01-31   pump  16630   dump   7389
  TEST  2026-02-01..2026-07-31   pump  14270   dump   6961


## 0. Проверка инструментов

Ни одно число ниже нельзя читать, пока движок не проверен. Прошлая сессия закрылась
именно на этом (FINDINGS §5.2/5.5/5.6).

In [2]:
# run_book должен воспроизводить run_dca в ноль, когда новые ограничения выключены
B = build_book_v4(PUMP, DUMP, "b05", "b05", filtered=False)
check_parity(B.head(20000), f_pump=0.05, f_dump=0.03)

# замкнутая форма: n непересекающихся сделок с постоянным pnl -> start*(1+f*pnl)^n
from _engine import run_dca
en = pd.date_range("2024-01-01", periods=200, freq="2h")
syn = pd.DataFrame({"sym": [f"S{i}" for i in range(200)], "entry": en,
                    "exit_ts": en + pd.Timedelta(hours=1), "pnl": np.full(200, 0.02),
                    "liq": np.full(200, 1e12), "stream": ["pump"]*200})
R = run_dca(syn, f_pump=0.05, monthly=0., reserve=0., cap=False, slipmodel=False)
exp = 1000*(1.001)**200
print(f"  closed form: engine ${R['final']:,.6f}  expected ${exp:,.6f}  "
      f"rel {abs(R['final']-exp)/exp:.2e}  {'PASS' if abs(R['final']-exp)/exp < 1e-9 else 'FAIL'}")

  parity vs run_dca: final 13,762.935483 / 13,762.935483  taken 11993/11993  skipped 8007/8007  PASS
  closed form: engine $1,221.280705  expected $1,221.280705  rel 2.23e-14  PASS


## 1. Где именно исследование завысило edge

Прошлая сессия назвала причиной «полный размер при усреднённой цене входа». Это было
неточно. Настоящая причина видна в одной строке `_build_signals.py:132`:

```python
ws = np.arange(1, len(pr)+1); ws /= ws.sum()      # <- нормировка на РЕАЛИЗОВАННОЕ k
```

Цена от нормировки не зависит (взвешенное среднее инвариантно к масштабу). А **итоговый
размер — зависит**: чтобы ставить транши ∝ i и получить в сумме ровно target, надо заранее
знать длину кластера. Реальный трейдер с таким расписанием получит размер ∝ k².

Проверка не рассуждением, а построением: расписание `un01` ставит транши `0.01·i` **без
кэпа** — это в точности ценовой профиль эталона, исполнимый каузально. Если диагноз верен,
у `un01` и `et_rs` должны совпасть per-trade доходность и стоп-рейт, и разойтись `frac`.

In [3]:
rows = []
for leg, D in [("pump", PUMP), ("dump", DUMP)]:
    for s in ["et_rs", "un01"]:
        p, f = D[f"pnl_{s}"], D[f"frac_{s}"]
        rows.append(dict(leg=leg, sched=s, mean=p.mean()*100, med=p.median()*100,
                         stop=D[f"stop_{s}"].mean()*100, frac_mean=f.mean()*100,
                         frac_max=f.max()*100, capwt=(f*p).sum()/f.sum()*100))
t = pd.DataFrame(rows)
print(t.round(3).to_string(index=False))
print()
for leg, D in [("pump", PUMP), ("dump", DUMP)]:
    d = (D.pnl_un01 - D.pnl_et_rs).abs()
    print(f"{leg}: |pnl_un01 - pnl_et_rs| max {d.max():.2e}, "
          f"identical on {(d<1e-12).mean()*100:.1f}% of events "
          f"(the rest = where the uncapped ramp keeps filling past et_rs's last tranche)")
print()
print("=> одинаковая доходность позиции, разный размер позиции. Эталон считал доходность")
print("   позиции размера ∝ k² так, как будто это позиция фиксированного размера.")

 leg sched  mean   med  stop  frac_mean  frac_max  capwt
pump et_rs 0.392 1.135 2.970     99.983     100.0  0.398
pump  un01 0.392 1.135 2.970     49.220    3570.0 -0.540
dump et_rs 2.015 1.661 8.201     99.549     100.0  2.138
dump  un01 2.015 1.661 8.201      4.055     190.0 -1.442

pump: |pnl_un01 - pnl_et_rs| max 8.88e-16, identical on 100.0% of events (the rest = where the uncapped ramp keeps filling past et_rs's last tranche)
dump: |pnl_un01 - pnl_et_rs| max 8.88e-16, identical on 100.0% of events (the rest = where the uncapped ramp keeps filling past et_rs's last tranche)

=> одинаковая доходность позиции, разный размер позиции. Эталон считал доходность
   позиции размера ∝ k² так, как будто это позиция фиксированного размера.


### 1.1 Разложение по длине кластера

`kfull` — сколько заливок расписание успевает сделать в живом кластере. Это та величина,
которая одновременно определяет и качество средней цены, и размер позиции.

In [4]:
def kbin(k): return pd.cut(k, [0,1,2,3,6,10**9], labels=["1","2","3","4-6","7+"])

for leg, D, SS in [("dump", DUMP, ["et_rs","first","b05","rn03"]),
                   ("pump", PUMP, ["et_rs","first","b05","a8b01"])]:
    for wn, w in [("TRAIN", TRAIN), ("VALID", VALID)]:
        X = window(D, w).copy(); X["kb"] = kbin(X.kfull)
        out = []
        for kb, g in X.groupby("kb", observed=True):
            r = {"k": kb, "n": len(g), "share%": round(len(g)/len(X)*100, 1)}
            for s in SS:
                r[s] = round((g[f"frac_{s}"]*g[f"pnl_{s}"]).mean()*100, 3)
            r["stop_first%"] = round(g.stop_first.mean()*100, 1)
            out.append(r)
        print(f"--- {leg} {wn}: mtu % by cluster depth ---")
        print(pd.DataFrame(out).to_string(index=False)); print()

--- dump TRAIN: mtu % by cluster depth ---
  k    n  share%  et_rs   first    b05    rn03  stop_first%
  1 3642    57.3  1.926   1.926  0.096   0.321          4.6
  2 1563    24.6  3.889   2.669  0.583   1.945          4.1
  3  809    12.7  7.038   3.018  2.111   7.038          7.0
4-6  304     4.8  6.191  -7.056  3.251   3.589         29.3
 7+   38     0.6 -4.470 -16.779 -8.861 -14.080         78.9

--- dump VALID: mtu % by cluster depth ---
  k    n  share%  et_rs   first     b05    rn03  stop_first%
  1 4566    61.8  1.767   1.767   0.088   0.294          6.1
  2 1387    18.8  2.093   1.320   0.314   1.046          9.4
  3  551     7.5  4.482   0.831   1.345   4.482          9.3
4-6  455     6.2  4.958  -9.242   3.633  -3.573         40.0
 7+  430     5.8 -5.839 -23.144 -22.817 -22.384         95.1

--- pump TRAIN: mtu % by cluster depth ---
  k    n  share%  et_rs  first    b05  a8b01  stop_first%
  1 4910    25.9 -0.036 -0.036 -0.002  0.003          0.8
  2 2566    13.6 -0.253  0.

**Что здесь написано.**

*Дамп.* Деньги лежат в `k=1..3` — это 89–95% событий, и там `first` даёт +1.4…+3.4%.
Бот (`b05`) кладёт туда **5% намеченного размера** (+0.07…+0.09% mtu на `k=1`) и **100%** —
в каскады `k≥4`, где теряет −23%. Это и есть мартингейл, в одной таблице.

*Разница TRAIN↔VALID — одна величина:* доля глубоких каскадов. На TRAIN `k≥7` = 0.6%
событий, на VALID = 5.8% (крах 2025Q4). Ничего не «сломалось» — сменился состав.

*Памп.* Расхождение возникает только на длинных кластерах, где `frac` уже 100% у всех,
то есть разница чисто в цене входа: при `k=16-30` вход первым траншем на ~6% ниже
эталонной средней, при `k=30+` — на ~15%.

## 2. Все каузальные расписания, TRAIN и VALID

`et`/`et_rs` оставлены как **верхняя граница, не как кандидат** — их размер неисполним.
`et_rs` отличается от `et` только честным стопом по бегущей средней (проверяется на
нетриггерных барах внутри кластера, как в `core.py:stop_price`).

In [5]:
from _build_signals_v4 import SCHEDULES

def sc(X, s):
    p = X[f"pnl_{s}"].values; f = X[f"frac_{s}"].values; m = f*p
    return dict(frac=f.mean()*100, stop=X[f"stop_{s}"].mean()*100, mean=p.mean()*100,
                med=np.median(p)*100, mtu=m.mean()*100,
                capwt=m.sum()/f.sum()*100 if f.sum() else np.nan)

for leg, D in [("dump", DUMP), ("pump", PUMP)]:
    for wn, w in [("TRAIN", TRAIN), ("VALID", VALID)]:
        X = window(D, w)
        t = pd.DataFrame({s: sc(X, s) for s in SCHEDULES}).T.sort_values("mtu", ascending=False)
        print(f"--- {leg} {wn}  n={len(X)} ---")
        print(t.round(3).to_string()); print()

--- dump TRAIN  n=6356 ---
           frac   stop   mean    med    mtu  capwt
et_rs    99.754  4.264  3.171  3.401  3.225  3.233
et      100.000  3.965  3.189  3.369  3.189  3.189
eq02     71.193  5.318  2.315  2.733  1.797  2.524
eq03     53.472  4.767  2.702  3.020  1.710  3.199
first   100.000  6.403  1.706  1.984  1.706  1.706
rn03     39.810  4.626  2.992  3.313  1.645  4.133
dc04     61.699  4.751  2.479  2.737  1.625  2.633
rt03     99.525  4.814  1.478  1.759  1.580  1.587
eq04     41.445  4.578  2.811  3.103  1.385  3.341
dc06     52.690  4.704  2.502  2.750  1.384  2.626
sq04     32.849  4.531  2.956  3.235  1.245  3.791
dc09     45.677  4.688  2.506  2.756  1.196  2.619
dc16     38.223  4.688  2.506  2.756  1.001  2.619
rc06     99.662  4.626  0.870  1.409  0.945  0.948
e10     100.000  4.610  0.930  1.500  0.930  0.930
eq06     28.120  4.500  2.846  3.123  0.921  3.274
rt06     99.437  4.563  0.713  1.183  0.835  0.840
sq06     19.266  4.452  2.999  3.274  0.704  3.653
rc12

### 2.1 Хвост: edge или лотерея

Среднее, которое несут несколько сделок, — не edge. Доля денег в верхнем 1% сделок
разделяет расписания жёстче, чем само среднее.

In [6]:
X = window(DUMP, TRAIN)
for s in ["et_rs", "first", "rn03", "b05", "e20", "e30"]:
    m = (X[f"frac_{s}"]*X[f"pnl_{s}"]).values; tot = m.sum(); o = np.sort(m)[::-1]
    print(f"  {s:6s} total {tot:8.2f}   top1 {o[0]/tot*100:5.1f}%   top10 {o[:10].sum()/tot*100:5.1f}%"
          f"   top1% {o[:len(o)//100].sum()/tot*100:6.1f}%   median*n {np.median(m)*len(m):8.2f}")
print("\n  'median*n' близкое к total => результат несёт типичная сделка, а не хвост.")

  et_rs  total   204.99   top1   1.2%   top10   5.2%   top1%   15.0%   median*n   216.20
  first  total   108.45   top1   2.2%   top10   9.6%   top1%   27.1%   median*n   126.09
  rn03   total   104.58   top1   2.3%   top10   6.5%   top1%   19.3%   median*n    44.08
  b05    total    36.22   top1   2.0%   top10   9.4%   top1%   27.0%   median*n    13.62
  e20    total    42.78   top1   5.3%   top10  22.5%   top1%   63.5%   median*n    69.55
  e30    total    32.43   top1   6.7%   top10  27.1%   top1%   78.6%   median*n    53.29

  'median*n' близкое к total => результат несёт типичная сделка, а не хвост.


In [7]:
# стабильность по кварталам
t = {}
for s in ["et_rs", "first", "rn03", "b05", "e20"]:
    t[s] = (DUMP[f"frac_{s}"]*DUMP[f"pnl_{s}"]).groupby(DUMP.entry.dt.to_period("Q")).mean()*100
t = pd.DataFrame(t); t["n"] = DUMP.groupby(DUMP.entry.dt.to_period("Q")).size()
print("dump: mtu % by quarter"); print(t.round(3).to_string())
print("\npositive quarters: " + ", ".join(f"{c} {int((t[c]>0).sum())}/{len(t)}" for c in t.columns[:-1]))

dump: mtu % by quarter
        et_rs  first   rn03    b05    e20     n
entry                                          
2024Q1  6.964  4.016  3.510  1.166  1.702  1076
2024Q2  3.348  0.676  1.920  0.699  1.642  1165
2024Q3  1.324  0.612  0.059  0.040 -0.256   580
2024Q4  3.783  3.216  1.859  0.575  0.399  1221
2025Q1  2.260  1.559  1.692  0.582  0.412  1323
2025Q2  0.735 -0.612 -0.098  0.058 -0.353   991
2025Q3  0.874 -0.198 -0.035 -0.013 -0.558  1715
2025Q4  1.828 -1.294 -1.406 -1.453  2.971  4606
2026Q1  2.780  1.525  0.296 -0.058  0.387  2815
2026Q2  1.373 -0.272 -0.104 -0.155 -0.581  3997
2026Q3  0.275 -1.585 -0.792 -0.221 -1.216  1217

positive quarters: et_rs 11/11, first 6/11, rn03 6/11, b05 6/11, e20 6/11


## 3. Памп-нога: ни денег, ни хеджа

README §3 утверждал, что памп даёт и заработок (через частоту), и хедж с корреляцией −0.5,
и что без него combined Sharpe рушится. Оба утверждения проверяются здесь при честном сайзинге.

In [8]:
for wn, w in [("TRAIN", TRAIN), ("VALID", VALID)]:
    X = window(PUMP, w)
    t = pd.DataFrame({s: sc(X, s) for s in
                      ["et_rs","un01","a8b01","ab08","b05","rn12","e90","e120","first"]}).T
    print(f"--- pump {wn}: только et_rs (неисполним) положителен ---")
    print(t.sort_values("mtu", ascending=False).round(3).to_string()); print()

--- pump TRAIN: только et_rs (неисполним) положителен ---
          frac   stop   mean    med    mtu  capwt
et_rs   99.986  1.320  0.038  0.658  0.044  0.044
un01    37.567  1.320  0.038  0.658 -0.017 -0.047
a8b01   13.220  0.005  0.132  0.035 -0.020 -0.150
ab08    48.496  0.005  0.091  0.050 -0.075 -0.155
rn12    30.416  1.563 -0.149  0.565 -0.108 -0.354
b05     52.052  1.764 -0.346  0.435 -0.291 -0.559
e120   100.000  0.449 -0.337 -0.105 -0.337 -0.337
e90    100.000  0.671 -0.401 -0.159 -0.401 -0.401
first  100.000  2.012 -0.691  0.133 -0.691 -0.691

--- pump VALID: только et_rs (неисполним) положителен ---
          frac   stop   mean    med    mtu  capwt
et_rs   99.971  4.095  0.138  1.254  0.148  0.148
a8b01   14.269  0.012  0.175  0.066 -0.049 -0.347
ab08    51.209  0.036  0.068  0.088 -0.210 -0.409
e120   100.000  1.479 -0.242  0.076 -0.242 -0.242
e90    100.000  2.177 -0.392  0.099 -0.392 -0.392
rn12    35.187  5.286 -0.467  1.041 -0.564 -1.603
un01    56.987  4.095  0.138  1.2

In [9]:
# хедж: дамп зафиксирован, вес пампа растёт
CB = build_book_v4(PUMP, DUMP, "a8b01", "first", filtered=True)
for wn, w in [("TRAIN", TRAIN), ("VALID", VALID)]:
    B = window(CB, w)
    print(f"--- {wn} ---")
    for wp in [0.0, 0.01, 0.02, 0.03, 0.05]:
        R = run_book(B, ffrac=ffrac(B, wp, 0.03), fixed_equity=True, monthly=0., stopcol="stopped")
        s = stats(R, "")
        print(f"   w_pump={wp:.2f}   ${s['money']:+7,.0f}   Sh {s['Sharpe']:5.2f}   "
              f"DD {s['maxDD']:6.1%}   n {s['taken']:5d}")
print("\n=> монотонно хуже и по деньгам, и по просадке, на обоих окнах. Нога снимается.")

--- TRAIN ---
   w_pump=0.00   $    +71   Sh  0.32   DD -29.3%   n  3536


   w_pump=0.01   $     +5   Sh  0.12   DD -31.9%   n  9864


   w_pump=0.02   $    -45   Sh -0.03   DD -33.8%   n 12643


   w_pump=0.03   $   -104   Sh -0.22   DD -36.3%   n 12643


   w_pump=0.05   $   -210   Sh -0.58   DD -41.1%   n 14703
--- VALID ---
   w_pump=0.00   $    -72   Sh  0.33   DD -52.1%   n  3578


   w_pump=0.01   $   -133   Sh  0.22   DD -53.8%   n  9398


   w_pump=0.02   $   -169   Sh  0.15   DD -54.6%   n 11809


   w_pump=0.03   $   -218   Sh  0.07   DD -56.6%   n 11816


   w_pump=0.05   $   -298   Sh -0.07   DD -59.1%   n 13575

=> монотонно хуже и по деньгам, и по просадке, на обоих окнах. Нога снимается.


## 4. Портфель при сопоставленном риске

Сравнивать расписания при одном весе ноги нельзя: то, что разворачивает 1/6 размера,
просто ведёт меньшую книгу. Поэтому вес подбирается под общий бюджет просадки на TRAIN.

In [10]:
SCH = ["et_rs", "first", "rn03", "rn06", "rn12", "b05", "un01", "e20"]
books = {}
for s in SCH:
    b = build_book_v4(PUMP, DUMP, "b05", s, filtered=True)
    books[s] = b[b.stream == "dump"]
rows = []
for wn, w in [("TRAIN", TRAIN), ("VALID", VALID)]:
    for s in SCH:
        B = window(books[s], w)
        for wd in [0.01, 0.02, 0.03, 0.05, 0.08, 0.12, 0.20, 0.35]:
            R = run_book(B, ffrac=ffrac(B, 0., wd), fixed_equity=True, monthly=0., stopcol="stopped")
            st = stats(R, s)
            rows.append(dict(win=wn, sched=s, w=wd, money=st["money"], sh=st["Sharpe"],
                             dd=st["maxDD"]*100, n=st["taken"]))
T = pd.DataFrame(rows)
tr = T[T.win == "TRAIN"]
sel = tr.loc[tr.groupby("sched").apply(lambda g: (g.dd + 20).abs().idxmin())]
out = []
for _, r in sel.iterrows():
    v = T[(T.win=="VALID") & (T.sched==r.sched) & (T.w==r.w)].iloc[0]
    out.append(dict(sched=r.sched, w=r.w, tr_money=round(r.money), tr_sh=round(r.sh,2),
                    tr_dd=round(r.dd,1), va_money=round(v.money), va_sh=round(v.sh,2),
                    va_dd=round(v.dd,1)))
print("вес подобран под TRAIN maxDD ~ -20%, затем перенесён на VALID без изменений:")
print(pd.DataFrame(out).sort_values("tr_sh", ascending=False).to_string(index=False))

вес подобран под TRAIN maxDD ~ -20%, затем перенесён на VALID без изменений:
sched    w  tr_money  tr_sh  tr_dd  va_money  va_sh  va_dd
et_rs 0.35     10241   3.40  -20.4      -935  -2.83  -95.1
 rn03 0.08       524   1.38  -17.8       160   0.79  -46.7
 rn06 0.20       540   1.26  -18.9      -287   0.14  -42.7
  b05 0.20       509   1.18  -19.4       100   0.75  -37.6
 rn12 0.35       209   0.66  -16.4      -322  -0.27  -50.0
first 0.02        45   0.27  -22.1       -40   0.15  -35.1
 un01 0.35        12   0.14  -20.9      -221  -0.25  -44.9
  e20 0.02       -82  -0.32  -20.3       -69  -0.10  -24.5


## 5. Что было отвергнуто на TRAIN

Каждый рычаг проверялся отдельно. Отвергнутое отвергнуто по TRAIN, а не по тому, что
оно не понравилось на VALID.

In [11]:
B0 = window(books["rn03"], TRAIN); B1 = window(books["rn03"], VALID)
def two(lab, **kw):
    a = stats(run_book(B0, ffrac=ffrac(B0,0.,0.10), fixed_equity=True, monthly=0., stopcol="stopped", **kw), "")
    b = stats(run_book(B1, ffrac=ffrac(B1,0.,0.10), fixed_equity=True, monthly=0., stopcol="stopped", **kw), "")
    print(f"  {lab:<34s} TRAIN ${a['money']:+7,.0f} Sh {a['Sharpe']:5.2f} DD {a['maxDD']:6.1%}"
          f" | VALID ${b['money']:+7,.0f} Sh {b['Sharpe']:5.2f} DD {b['maxDD']:6.1%}")
two("baseline rn03 @10%")
two("1 позиция на символ", sym_slots=1)
two("gross на символ <= 6% eq", sym_gross=0.06)
two("cooldown 240м после стопа", cooldown_min=240)
two("MAX слотов 6", MAX=6)
two("MAX слотов 40", MAX=40)

  baseline rn03 @10%                 TRAIN $   +557 Sh  1.30 DD -21.3% | VALID $   +302 Sh  0.98 DD -49.7%


  1 позиция на символ                TRAIN $   +293 Sh  0.88 DD -19.1% | VALID $   -128 Sh  0.34 DD -45.7%


  gross на символ <= 6% eq           TRAIN $   +400 Sh  1.18 DD -18.7% | VALID $   +230 Sh  0.90 DD -39.1%


  cooldown 240м после стопа          TRAIN $   +536 Sh  1.22 DD -18.7% | VALID $    -36 Sh  0.68 DD -60.3%


  MAX слотов 6                       TRAIN $   +151 Sh  0.52 DD -30.3% | VALID $   +179 Sh  0.77 DD -43.5%


  MAX слотов 40                      TRAIN $   +668 Sh  1.37 DD -20.9% | VALID $   +282 Sh  0.96 DD -49.6%


In [12]:
# уровень катастроф-стопа — отдельная сборка (_build_stopsweep_v4.py)
SS = pd.read_parquet("_out/dump_stopsweep.parquet")
SS["entry"] = pd.to_datetime(SS.entry).dt.tz_localize(None)
tr = SS[SS.entry < "2025-07-01"]
stops = ["05","08","10","12","15","20","inf"]
out = {}
for s in ["first","b05","rn03","et_rs"]:
    out[s] = {st: round((tr[f"frac_{s}__{st}"]*tr[f"pnl_{s}__{st}"]).mean()*100, 3) for st in stops}
print("dump, TRAIN, mtu % по уровню катастроф-стопа (текущий = 20%):")
print(pd.DataFrame(out).T.to_string())
print("\n=> чем ТУЖЕ стоп, тем хуже, монотонно и на всех расписаниях. Гипотеза «стоп на -8%")
print("   срежет каскад» опровергнута: убыток на k>=4 создаётся ЦЕНОЙ ВХОДА, а не выходом.")

dump, TRAIN, mtu % по уровню катастроф-стопа (текущий = 20%):
          05     08     10     12     15     20    inf
first -0.283  0.474  0.744  0.942  1.255  1.706  2.346
b05    0.125  0.208  0.326  0.435  0.510  0.570  0.595
rn03   0.380  0.526  0.864  1.194  1.479  1.645  1.772
et_rs  1.643  2.005  2.377  2.736  2.992  3.225  3.589

=> чем ТУЖЕ стоп, тем хуже, монотонно и на всех расписаниях. Гипотеза «стоп на -8%
   срежет каскад» опровергнута: убыток на k>=4 создаётся ЦЕНОЙ ВХОДА, а не выходом.


## 6. TEST — один прогон

Конфигурация заморожена по §2–§5 до того, как это окно открылось:
дамп-only, расписание `rn03`, вес 10% (под бюджет −20% просадки на TRAIN),
walk-forward классификатор включён, стоп −20%, MAX 18.

In [13]:
CAND = books["rn03"]; BOUND = books["et_rs"]
TODAY = build_book_v4(PUMP, DUMP, "b05", "b05", filtered=True)
def row(book, wp, wd, w, lab, uni=None):
    B = book if uni is None else book[book.sym.isin(uni)]
    B = window(B, w)
    R = run_book(B, ffrac=ffrac(B, wp, wd), fixed_equity=True, monthly=0., stopcol="stopped")
    s = stats(R, lab)
    print(f"  {lab:<42s} ${s['money']:+8,.0f}  Sh {s['Sharpe']:5.2f}  DD {s['maxDD']:6.1%}  "
          f"worst day {s['worstday']:6.1%}  n {s['taken']:5d}")
for wn, w in [("TRAIN", TRAIN), ("VALID", VALID), ("TEST", TEST)]:
    print(f"===== {wn}  {w[0]}..{w[1]} =====")
    row(TODAY, 0.05, 0.03, w, "бот сегодня (памп b05 5% + дамп b05 3%)")
    row(CAND, 0., 0.10, w, "КАНДИДАТ дамп-only rn03 @10%")
    row(CAND, 0., 0.10, w, "КАНДИДАТ на 120 символах бота", uni=BOT120)
    row(BOUND, 0., 0.08, w, "et_rs @8% (неисполним, верхняя граница)")
    print()

===== TRAIN  2024-01-02..2025-06-30 =====


  бот сегодня (памп b05 5% + дамп b05 3%)    $    -174  Sh  0.06  DD -63.5%  worst day -14.0%  n 16103
  КАНДИДАТ дамп-only rn03 @10%               $    +557  Sh  1.30  DD -21.3%  worst day -10.5%  n  3509
  КАНДИДАТ на 120 символах бота              $    +485  Sh  1.46  DD  -7.2%  worst day  -7.0%  n  1011
  et_rs @8% (неисполним, верхняя граница)    $  +3,013  Sh  3.19  DD -16.4%  worst day -14.9%  n  3490

===== VALID  2025-07-01..2026-01-31 =====


  бот сегодня (памп b05 5% + дамп b05 3%)    $    -484  Sh -0.41  DD -67.4%  worst day -19.4%  n 11921
  КАНДИДАТ дамп-only rn03 @10%               $    +302  Sh  0.98  DD -49.7%  worst day -40.3%  n  4657
  КАНДИДАТ на 120 символах бота              $    -213  Sh -0.23  DD -48.7%  worst day -40.0%  n   892


  et_rs @8% (неисполним, верхняя граница)    $  +5,477  Sh  4.63  DD -25.1%  worst day -17.0%  n  4793

===== TEST  2026-02-01..2026-07-31 =====


  бот сегодня (памп b05 5% + дамп b05 3%)    $    -863  Sh -1.70  DD -94.0%  worst day -34.2%  n 11040
  КАНДИДАТ дамп-only rn03 @10%               $    -927  Sh -1.05  DD -93.3%  worst day -34.9%  n  4368
  КАНДИДАТ на 120 символах бота              $    -457  Sh -0.99  DD -61.0%  worst day -14.3%  n  1720


  et_rs @8% (неисполним, верхняя граница)    $  +4,268  Sh  4.47  DD -12.2%  worst day  -9.2%  n  6018



In [14]:
B = window(CAND, TEST)
R = run_book(B, ffrac=ffrac(B, 0., 0.10), fixed_equity=True, monthly=0., stopcol="stopped")
tk = R["tk"]
g = tk.groupby(tk.entry.dt.to_period("M")).agg(n=("usd","size"), money=("usd","sum"),
                                               mean_pnl=("pnl","mean"))
g["mean_pnl"] *= 100
print("кандидат на TEST, по месяцам:"); print(g.round(2).to_string())
print(f"\nсредняя на сделку положительна в {(g.mean_pnl>0).sum()} месяцах из {len(g)}, "
      f"а деньги — в {(g.money>0).sum()}. Это та же связка размер-неблагоприятность:")
print("даже у rn03 разброс размера 6x между k=1 и k>=3.")

кандидат на TEST, по месяцам:
            n   money  mean_pnl
entry                          
2026-02   641 -698.30     -0.69
2026-03   497   50.23      0.77
2026-04  1196  354.18      0.97
2026-05   709 -166.93      0.46
2026-06   868 -320.64      0.16
2026-07   457 -145.79     -0.34

средняя на сделку положительна в 4 месяцах из 6, а деньги — в 2. Это та же связка размер-неблагоприятность:
даже у rn03 разброс размера 6x между k=1 и k>=3.


## 7. Можно ли обойти нормировку: три хода, все закрыты

Эталон исполним ровно в одном смысле — его ЦЕНОВОЙ профиль воспроизводится траншами ∝ i
(`un01`, §1). Неисполнима нормировка: чтобы итог был равен target при любом k, первый транш
должен быть `1/(1+2+...+k)`, то есть зависеть от будущей длины кластера. Ниже — три попытки
обойти это, каждая закрыта своим измерением.

In [15]:
# ход 1: предсказать k по бару-триггера. Тогда нормировка становится исполнимой.
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.metrics import r2_score, roc_auc_score
from _engine import PF, DF
for leg, D, F, thr in [("dump", DUMP, DF, 4), ("pump", PUMP, PF, 9)]:
    D = D.sort_values("entry").reset_index(drop=True)
    y = np.log1p(D.kfull.values); deep = (D.kfull >= thr).values; X = D[F].values
    cuts = [int(len(D)*q) for q in (0.40,0.55,0.70,0.85,1.0)]; prev = cuts[0]
    pr = np.full(len(D), np.nan); pc = np.full(len(D), np.nan)
    for cut in cuts[1:]:
        pr[prev:cut] = HistGradientBoostingRegressor(max_depth=3, learning_rate=0.05,
            max_iter=300, l2_regularization=1.0, min_samples_leaf=50,
            random_state=0).fit(X[:prev], y[:prev]).predict(X[prev:cut])
        pc[prev:cut] = HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05,
            max_iter=300, l2_regularization=1.0, min_samples_leaf=50,
            random_state=0).fit(X[:prev], deep[:prev]).predict_proba(X[prev:cut])[:,1]
        prev = cut
    ok = ~np.isnan(pr)
    print(f"{leg}: walk-forward R2 on log(kfull) {r2_score(y[ok], pr[ok]):+.4f}   "
          f"AUC 'kfull>={thr}' {roc_auc_score(deep[ok], pc[ok]):.4f}   P(deep)={deep.mean()*100:.1f}%")
print()
print("=> у дампа глубина НЕ предсказуема (R2 отрицателен, AUC = монетка), у пампа предсказуема,")
print("   но у пампа нет edge, который стоило бы защищать. Ход 1 закрыт.")

dump: walk-forward R2 on log(kfull) -0.0174   AUC 'kfull>=4' 0.5153   P(deep)=9.4%


pump: walk-forward R2 on log(kfull) +0.3046   AUC 'kfull>=9' 0.7840   P(deep)=28.9%

=> у дампа глубина НЕ предсказуема (R2 отрицателен, AUC = монетка), у пампа предсказуема,
   но у пампа нет edge, который стоило бы защищать. Ход 1 закрыт.


In [16]:
# ход 2: не смешивать кластер в одну позицию — каждая ступень отдельной сделкой
# фиксированного размера со СВОИМ стопом (_build_steps_v4.py)
S = pd.read_parquet("_out/dump_steps.parquet")
S["entry"] = pd.to_datetime(S.entry)
if S.entry.dt.tz is not None: S["entry"] = S.entry.dt.tz_localize(None)
for wn, w in [("TRAIN", TRAIN), ("VALID", VALID), ("TEST", TEST)]:
    X = window(S, w); s14 = X[X.step <= 4]; E = window(DUMP, w)
    g = X.groupby(X.step.clip(upper=4)).agg(n=("pnl","size"), mean=("pnl","mean"),
                                            stop=("stopped","mean"))
    print(f"{wn}: steps1-4 mean {s14.pnl.mean()*100:+.3f}%   "
          f"et_rs mean {E.pnl_et_rs.mean()*100:+.3f}%   "
          f"step stop-rate by step {[round(v*100,1) for v in g['stop']]}   "
          f"et_rs stop {E.stop_et_rs.mean()*100:.1f}%")
print()
print("=> ход 2 закрыт, и он опровергает интуицию: усреднение защищает не глубокие входы,")
print("   а МЕЛКИЕ. Стоп первого транша стоит выше всех и срабатывает первым; в смешанной")
print("   позиции его убыток подпирается дешёвыми доливками. Разбив кластер, снимаешь защиту.")

TRAIN: steps1-4 mean +3.277%   et_rs mean +3.171%   step stop-rate by step [6.4, 4.3, 5.1, 19.3]   et_rs stop 4.3%
VALID: steps1-4 mean -1.440%   et_rs mean +1.576%   step stop-rate by step [14.3, 24.2, 33.7, 38.8]   et_rs stop 10.2%
TEST: steps1-4 mean -0.308%   et_rs mean +1.425%   step stop-rate by step [12.9, 17.6, 24.3, 40.1]   et_rs stop 9.6%

=> ход 2 закрыт, и он опровергает интуицию: усреднение защищает не глубокие входы,
   а МЕЛКИЕ. Стоп первого транша стоит выше всех и срабатывает первым; в смешанной
   позиции его убыток подпирается дешёвыми доливками. Разбив кластер, снимаешь защиту.


In [17]:
# ход 3: форма транша. Насколько быстро размер должен расти с глубиной?
SH = ["et_rs","first","dc04","dc16","eq02","eq04","eq09","sq04","rn03","rn06","b05"]
for wn, w in [("TRAIN", TRAIN), ("TEST", TEST)]:
    X = window(DUMP, w); rows = []
    for s in SH:
        p = X[f"pnl_{s}"].values; f = X[f"frac_{s}"].values
        fk = X.groupby(X.kfull.clip(upper=6))[f"frac_{s}"].mean()
        rows.append(dict(sched=s, frac=f.mean()*100, spread=fk.max()/max(fk.min(),1e-9),
                         mean=p.mean()*100, mtu=(f*p).mean()*100))
    print(f"-- dump {wn}: 'spread' = во сколько раз размер на k=6 больше, чем на k=1 --")
    print(pd.DataFrame(rows).set_index("sched").round(3).to_string()); print()
print("=> положителен по mtu только et_rs, и он единственный со spread ~1 ПРИ усреднении.")
print("   `first` тоже плоский (spread 1.0), но не усредняет — и теряет.")
print("   Всё, что между, теряет. Ход 3 закрыт.")

-- dump TRAIN: 'spread' = во сколько раз размер на k=6 больше, чем на k=1 --
          frac  spread   mean    mtu
sched                               
et_rs   99.754   1.007  3.171  3.225
first  100.000   1.000  1.706  1.706
dc04    61.699   2.083  2.479  1.625
dc16    38.223   2.567  2.506  1.001
eq02    71.193   2.000  2.315  1.797
eq04    41.445   4.000  2.811  1.385
eq09    18.873   7.111  2.853  0.604
sq04    32.849   6.146  2.956  1.245
rn03    39.810   6.000  2.992  1.645
rn06    13.151  21.000  3.149  0.544
b05     13.759  20.000  3.145  0.570

-- dump TEST: 'spread' = во сколько раз размер на k=6 больше, чем на k=1 --
          frac  spread   mean    mtu
sched                               
et_rs   99.700   1.067  1.425  1.491
first  100.000   1.000 -0.175 -0.175
dc04    62.988   2.082  0.415 -0.163
dc16    39.553   2.603  0.534 -0.121
eq02    72.238   2.000  0.363 -0.090
eq04    43.499   3.994  0.759 -0.160
eq09    21.076   7.409  0.978 -0.151
sq04    35.288   6.134  0.913 -0

## 7. Выводы

Считаются ячейкой ниже, чтобы не разойтись с таблицами.

In [18]:
print("1. ПРИЧИНА РАСХОЖДЕНИЯ ИССЛЕДОВАНИЯ И БОТА — НАЙДЕНА И ДОКАЗАНА ПОСТРОЕНИЕМ.")
print("   `_build_signals.py:132` нормирует веса траншей на РЕАЛИЗОВАННУЮ длину кластера.")
print("   Цена от этого не зависит, размер — зависит. `un01` воспроизводит эталонную")
print("   доходность в ноль, разворачивая капитал ∝ k². Прошлая формулировка ('полный")
print("   размер при усреднённой цене') была неточной: дело в нормировке.")
print()
print("2. ПАМП-НОГА СНИМАЕТСЯ. Отрицательна при каждом каузальном расписании на TRAIN и")
print("   VALID; как хедж монотонно ухудшает и деньги, и просадку при росте веса 0->5%")
print("   на обоих окнах. Тезис 'без пампа Sharpe рушится' держался на завышенном сайзинге.")
print()
print("3. СИГНАЛ В ДАМПЕ РЕАЛЕН И НЕ ВЫДОХСЯ. et_rs положителен в 11 кварталах из 11 и на")
print("   всех трёх окнах (Sharpe 3.2 / 4.6 / 4.5). Проблема НЕ в сигнале.")
print()
print("4. НО ИСПОЛНИМОГО РАЗМЕЩЕНИЯ КАПИТАЛА НЕТ, И ЭТО СТРУКТУРНО.")
print("   Работает только 'плоский размер + усреднение'. Усреднение вниз — это по")
print("   определению добавление размера на падении, поэтому плоский итог достижим лишь")
print("   если НАЧИНАТЬ МЕЛКО, КОГДА КЛАСТЕР ОКАЖЕТСЯ ГЛУБОКИМ. Это и есть знание k.")
print("   Замкнуто с трёх сторон: k непредсказуем (§7 ход 1, AUC 0.515); разбиение на")
print("   независимые сделки снимает защиту мелких входов (ход 2); ни одна форма транша")
print("   от ∝1/i до ∝i не проходит все три окна (ход 3).")
print()
print("5. ОТВЕРГНУТО НА TRAIN: тугой стоп (монотонно хуже на всех расписаниях), кап на")
print("   символ, одна позиция на символ, cooldown после стопа, ликвидностный фильтр,")
print("   изменение лимита слотов, вход по фиксированному клоку (eNN — хвостовой и")
print("   нестабильный: 79% денег TRAIN в верхнем 1% сделок).")
print()
print("6. ЧЕГО ЭТА СЕССИЯ НЕ ТРОГАЛА (и где остаётся пространство):")
print("   - ГОРИЗОНТ ВЫХОДА. Все прогоны держат 240 минут от последнего триггера. Ни разу")
print("     не варьировался, хотя это ровно та ручка, которая решает, дожил ли ты до отскока.")
print("   - ПОРОГ СОБЫТИЯ (-7%/15м) и определение кластера (склейка через 10 тихих минут).")
print("   - Размер как функция ВОЛАТИЛЬНОСТИ символа, а не глубины кластера.")
print("   Их стоит пробовать раньше, чем возвращаться к форме транша: она закрыта.")

1. ПРИЧИНА РАСХОЖДЕНИЯ ИССЛЕДОВАНИЯ И БОТА — НАЙДЕНА И ДОКАЗАНА ПОСТРОЕНИЕМ.
   `_build_signals.py:132` нормирует веса траншей на РЕАЛИЗОВАННУЮ длину кластера.
   Цена от этого не зависит, размер — зависит. `un01` воспроизводит эталонную
   доходность в ноль, разворачивая капитал ∝ k². Прошлая формулировка ('полный
   размер при усреднённой цене') была неточной: дело в нормировке.

2. ПАМП-НОГА СНИМАЕТСЯ. Отрицательна при каждом каузальном расписании на TRAIN и
   VALID; как хедж монотонно ухудшает и деньги, и просадку при росте веса 0->5%
   на обоих окнах. Тезис 'без пампа Sharpe рушится' держался на завышенном сайзинге.

3. СИГНАЛ В ДАМПЕ РЕАЛЕН И НЕ ВЫДОХСЯ. et_rs положителен в 11 кварталах из 11 и на
   всех трёх окнах (Sharpe 3.2 / 4.6 / 4.5). Проблема НЕ в сигнале.

4. НО ИСПОЛНИМОГО РАЗМЕЩЕНИЯ КАПИТАЛА НЕТ, И ЭТО СТРУКТУРНО.
   Работает только 'плоский размер + усреднение'. Усреднение вниз — это по
   определению добавление размера на падении, поэтому плоский итог достижим лишь